In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("../datasets/processed/merged_data.csv")

df = df.drop(columns=["country_code"])

# TODO Seamus rec: convert quantile threshold to dynamic value set by user input
threshold = df["asylum_applications"].quantile(0.70)
df["high_asylum"] = (df["asylum_applications"] > threshold).astype(int)

#print(df.head())

X = df.drop(columns=["asylum_applications", "high_asylum", "year"])
y = df["high_asylum"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# TODO test iterations for better performance
model = LogisticRegression(max_iter=500)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False)

print("\nFeature importance:\n", coef_df)

#do not put year
# TODO correlation matrix/heatmap for features

Accuracy: 0.8157894736842105

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.91      0.87        53
           1       0.74      0.61      0.67        23

    accuracy                           0.82        76
   macro avg       0.79      0.76      0.77        76
weighted avg       0.81      0.82      0.81        76


Feature importance:
              feature  coefficient
2         population     1.720785
3          urban_pct     1.522808
8           dry_days     0.658429
6       precip_total     0.582643
5      heatwave_days     0.544911
1  unemployment_rate    -0.003125
4          temp_mean    -0.096121
7  precip_days_heavy    -0.115707
0     gdp_per_capita    -0.324406
9   evapotrans_total    -0.618497
